<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# NRMS: Neural News Recommendation with Multi-Head Self-Attention
NRMS \[1\] is a neural news recommendation approach with multi-head selfattention. The core of NRMS is a news encoder and a user encoder. In the newsencoder, a multi-head self-attentions is used to learn news representations from news titles by modeling the interactions between words. In the user encoder, we learn representations of users from their browsed news and use multihead self-attention to capture the relatedness between the news. Besides, we apply additive
attention to learn more informative news and user representations by selecting important words and news.

## Properties of NRMS:
- NRMS is a content-based neural news recommendation approach.
- It uses multi-self attention to learn news representations by modeling the iteractions between words and learn user representations by capturing the relationship between user browsed news.
- NRMS uses additive attentions to learn informative news and user representations by selecting important words and news.

## CPU first run and input format

The default uses invented articles, users and impressions with random 16-dimensional embeddings. It checks model execution with reduced dimensions; real MIND performance is unmeasured. [Installation, trusted prepared inputs and results](NEWS_CPU_VALIDATION.md).

The TSV layout follows the [MIND format](https://github.com/msnews/msnews.github.io/blob/master/assets/doc/introduction.md). `news.tsv` has eight fields: ID, category, subcategory, title, abstract, URL, title entities and abstract entities. `behaviors.tsv` has impression ID, user ID, timestamp, clicked history and candidate-label pairs. The fixture has 32 articles, 8 users, 32 train and 16 validation impressions, each with one positive and four negatives. Set `MIND_DATA_DIR` to a trusted prepared train/valid/utils folder; dictionary IDs must match its embedding rows.


## Global settings and imports

In [ ]:
import os
import sys
import numpy as np
import zipfile
from tqdm import tqdm
from tempfile import TemporaryDirectory
import tensorflow as tf
tf.get_logger().setLevel('ERROR') # only show error messages

from recommenders.models.deeprec.deeprec_utils import download_deeprec_resources 
from recommenders.models.newsrec.newsrec_utils import prepare_hparams
from recommenders.models.newsrec.models.nrms import NRMSModel
from recommenders.models.newsrec.io.mind_iterator import MINDIterator
from recommenders.models.newsrec.newsrec_utils import get_mind_data_set
from recommenders.utils.notebook_utils import store_metadata

print("System version: {}".format(sys.version))
print("Tensorflow version: {}".format(tf.__version__))

from recommenders.models.newsrec.quickstart import make_synthetic_news, require_news_files

import random
random.seed(42)


## CPU parameters
One epoch and batch size 8 exercise the full architecture with reduced dimensions. Set `NEWS_EPOCHS` before launching Python to increase training. This is a smoke test with synthetic inputs, not the original paper configuration.


In [ ]:
epochs = int(os.environ.get("NEWS_EPOCHS", "1"))
seed = 42
batch_size = 8
assert epochs > 0


## Input selection
Without `MIND_DATA_DIR`, create 32 invented articles, 8 users, 32 training and 16 validation impressions in a temporary directory. With `MIND_DATA_DIR`, read a trusted prepared MIND train/valid/utils folder without downloading or modifying it. Supply its original YAML and compatible dictionaries/embeddings. No real MIND run is claimed here.


In [ ]:
tmpdir = TemporaryDirectory()
prepared = os.environ.get("MIND_DATA_DIR")
if prepared:
    data_path = require_news_files(prepared, ['train/news.tsv', 'train/behaviors.tsv', 'valid/news.tsv', 'valid/behaviors.tsv', 'utils/nrms.yaml', 'utils/uid2index.pkl', 'utils/embedding.npy', 'utils/word_dict.pkl'])
    data_kind = "local_prepared_unverified_provenance"
else:
    data_path = str(make_synthetic_news(tmpdir.name))
    data_kind = "synthetic"
print("Input kind:", data_kind, "| CPU devices:", tf.config.list_physical_devices("CPU"), "| GPU devices:", tf.config.list_physical_devices("GPU"))
store_metadata("data_kind", data_kind)

train_news_file = os.path.join(data_path, 'train', 'news.tsv')
train_behaviors_file = os.path.join(data_path, 'train', 'behaviors.tsv')
valid_news_file = os.path.join(data_path, 'valid', 'news.tsv')
valid_behaviors_file = os.path.join(data_path, 'valid', 'behaviors.tsv')
wordEmb_file = os.path.join(data_path, "utils", "embedding.npy")
userDict_file = os.path.join(data_path, "utils", "uid2index.pkl")
wordDict_file = os.path.join(data_path, "utils", "word_dict.pkl")
yaml_file = os.path.join(data_path, "utils", 'nrms.yaml')


store_metadata("gpu_count", len(tf.config.list_physical_devices("GPU")))


## Create hyper-parameters

In [ ]:
hparams = prepare_hparams(yaml_file, 
                          wordEmb_file=wordEmb_file,
                          wordDict_file=wordDict_file, 
                          userDict_file=userDict_file,
                          batch_size=batch_size,
                          epochs=epochs,
                          show_step=10)
print(hparams)

## Train the NRMS model

In [ ]:
iterator = MINDIterator

In [ ]:
model = NRMSModel(hparams, iterator, seed=seed)

In [ ]:
print(model.run_eval(valid_news_file, valid_behaviors_file))

In [ ]:
%%time
model.fit(train_news_file, train_behaviors_file, valid_news_file, valid_behaviors_file)

In [ ]:
%%time
res_syn = model.run_eval(valid_news_file, valid_behaviors_file)
print(res_syn)


In [ ]:
# Record results for tests - ignore this cell
store_metadata("group_auc", res_syn['group_auc'])
store_metadata("mean_mrr", res_syn['mean_mrr'])
store_metadata("ndcg@5", res_syn['ndcg@5'])
store_metadata("ndcg@10", res_syn['ndcg@10'])

## Save the model

In [ ]:
model_path = os.path.join(tmpdir.name, "model")
os.makedirs(model_path, exist_ok=True)

model.model.save_weights(os.path.join(model_path, "nrms_ckpt"))
model.model.load_weights(os.path.join(model_path, "nrms_ckpt"))
reloaded = model.run_eval(valid_news_file, valid_behaviors_file)
assert all(np.isclose(res_syn[k], reloaded[k]) for k in res_syn)
store_metadata("reload_metrics_equal", True)


## Ranked predictions
Write one candidate ranking per validation impression in a temporary ZIP. This example does not upload predictions or submit a MIND challenge entry.


In [ ]:
group_impr_indexes, group_labels, group_preds = model.run_fast_eval(valid_news_file, valid_behaviors_file)

In [ ]:
with open(os.path.join(tmpdir.name, 'prediction.txt'), 'w') as f:
    for impr_index, preds in tqdm(zip(group_impr_indexes, group_preds)):
        impr_index += 1
        pred_rank = (np.argsort(np.argsort(preds)[::-1]) + 1).tolist()
        pred_rank = '[' + ','.join([str(i) for i in pred_rank]) + ']'
        f.write(' '.join([str(impr_index), pred_rank])+ '\n')

In [ ]:
f = zipfile.ZipFile(os.path.join(tmpdir.name, 'prediction.zip'), 'w', zipfile.ZIP_DEFLATED)
f.write(os.path.join(tmpdir.name, 'prediction.txt'), arcname='prediction.txt')
f.close()

## Reference
\[1\] Wu et al. "Neural News Recommendation with Multi-Head Self-Attention." in Proceedings of the 2019 Conference on Empirical Methods in Natural Language Processing and the 9th International Joint Conference on Natural Language Processing (EMNLP-IJCNLP)<br>
\[2\] Wu, Fangzhao, et al. "MIND: A Large-scale Dataset for News Recommendation" Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics. https://msnews.github.io/competition.html <br>
\[3\] GloVe: Global Vectors for Word Representation. https://nlp.stanford.edu/projects/glove/

In [ ]:
tmpdir.cleanup()
